## **Name: Chirag Chaudhari**
## **Roll no: 24102A2003**
## **Branch: CMPN-A**
## **Title:  Retail Transaction Analysis with R**


In [1]:

required_packages <- c("dplyr", "readr", "jsonlite", "readxl", "writexl",
                       "DBI", "RSQLite", "lubridate", "tibble")

not_installed <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(not_installed) > 0) {
  install.packages(not_installed, repos = "https://cloud.r-project.org")
}

invisible(lapply(required_packages, library, character.only = TRUE))
cat("Analysis libraries are ready.\n")


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Analysis libraries are ready.


##  Obtain and prepare source data


In [2]:
set.seed(2026)

retail_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
download_path <- tempfile(fileext = ".xlsx")

retail_source <- tryCatch({
  download.file(retail_url, download_path, mode = "wb", quiet = TRUE)
  if (file.exists(download_path) && file.size(download_path) > 1000) {
    readxl::read_excel(download_path)
  } else {
    stop("The downloaded workbook is empty.")
  }
}, error = function(e) {
  message("Could not retrieve the online workbook; using generated example data.")
  row_count <- 3200
  region_choices <- c("United Kingdom", "Germany", "France", "EIRE",
                      "Spain", "Netherlands", "Belgium", "Australia")
  tibble::tibble(
    InvoiceNo = sample(51000:51600, row_count, replace = TRUE),
    StockCode = paste0("P", sample(100:220, row_count, replace = TRUE)),
    Description = paste("Retail item", sample(100:220, row_count, replace = TRUE)),
    Quantity = sample(c(-3, 0, 1:40), row_count, replace = TRUE),
    InvoiceDate = as.character(sample(
      seq(as.Date("2011-01-01"), as.Date("2011-12-31"), by = "day"),
      row_count, replace = TRUE
    )),
    UnitPrice = round(runif(row_count, min = 0, max = 80), 2),
    CustomerID = sample(c(NA, 13000:13250), row_count, replace = TRUE),
    Country = sample(region_choices, row_count, replace = TRUE)
  )
})

# Standardize the core fields needed by the later steps.
retail_source <- as.data.frame(retail_source)
names(retail_source) <- trimws(names(retail_source))
head(retail_source)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
,<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
1,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
2,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
3,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
6,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


## Create three source files



In [3]:
transaction_export <- retail_source |>
  dplyr::select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate) |>
  dplyr::distinct()

product_export <- retail_source |>
  dplyr::select(StockCode, Description, UnitPrice) |>
  dplyr::distinct(StockCode, .keep_all = TRUE)

customer_export <- retail_source |>
  dplyr::filter(!is.na(CustomerID)) |>
  dplyr::select(CustomerID, Country) |>
  dplyr::distinct(CustomerID, .keep_all = TRUE)

readr::write_csv(transaction_export, "sales_transactions.csv")
jsonlite::write_json(product_export, "catalog_items.json", dataframe = "rows", auto_unbox = TRUE)
writexl::write_xlsx(customer_export, "customer_directory.xlsx")

cat("Files created:",
    "\nTransactions:", nrow(transaction_export),
    "\nProducts:", nrow(product_export),
    "\nCustomers:", nrow(customer_export), "\n")


Files created: 
Transactions: 536480 
Products: 4070 
Customers: 4372 


## Import and clean the files


In [4]:
sales_lines <- readr::read_csv("sales_transactions.csv", show_col_types = FALSE)
item_catalog <- jsonlite::fromJSON("catalog_items.json") |> tibble::as_tibble()
customer_directory <- readxl::read_excel("customer_directory.xlsx")

# A small helper keeps the customer table's key unique and usable.
clean_customer_directory <- function(x) {
  x |>
    dplyr::filter(!is.na(CustomerID), !is.na(Country)) |>
    dplyr::distinct(CustomerID, .keep_all = TRUE)
}

sales_lines_ok <- sales_lines |>
  dplyr::filter(!is.na(InvoiceNo), !is.na(StockCode), !is.na(CustomerID)) |>
  dplyr::distinct() |>
  dplyr::filter(Quantity > 0)

item_catalog_ok <- item_catalog |>
  dplyr::filter(!is.na(StockCode), !is.na(UnitPrice), UnitPrice > 0) |>
  dplyr::distinct(StockCode, .keep_all = TRUE)

customer_directory_ok <- clean_customer_directory(customer_directory)

cat("Rows before cleaning:",
    "\n  Transactions:", nrow(sales_lines),
    "\n  Products:", nrow(item_catalog),
    "\n  Customers:", nrow(customer_directory),
    "\nRows after cleaning:",
    "\n  Transactions:", nrow(sales_lines_ok),
    "\n  Products:", nrow(item_catalog_ok),
    "\n  Customers:", nrow(customer_directory_ok), "\n")


Rows before cleaning: 
  Transactions: 536480 
  Products: 4070 
  Customers: 4372 
Rows after cleaning: 
  Transactions: 392708 
  Products: 3855 
  Customers: 4372 


## Integrate the data sources



In [5]:
retail_analysis <- sales_lines_ok |>
  dplyr::inner_join(item_catalog_ok, by = "StockCode") |>
  dplyr::left_join(customer_directory_ok, by = "CustomerID") |>
  dplyr::mutate(LineRevenue = Quantity * UnitPrice)

cat("Integrated table:", nrow(retail_analysis), "rows by",
    ncol(retail_analysis), "columns\n")
cat("Rows without matched country:",
    sum(is.na(retail_analysis$Country)), "\n")

dplyr::glimpse(retail_analysis)


Integrated table: 387877 rows by 9 columns
Rows without matched country: 0 
Rows: 387,877
Columns: 9
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <dbl> 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <dttm> 2010-12-01 08:26:00, 2010-12-01 08:26:00, 2010-12-01 08:2…
$ Description <chr> "WHITE HANGING HEART T-LIGHT HOLDER", "WHITE METAL LANTERN…
$ UnitPrice   <dbl> 2.55, 3.39, 2.75, 3.39, 3.39, 7.65, 4.25, 1.85, 1.85, 1.69…
$ Country     <chr> "United Kingdom", "United Kingdom", "United Kingdom", "Uni…
$ LineRevenue <dbl> 15.30, 20.34, 22.00, 20.34, 20.34, 15.30, 25.50, 11.10, 11…


## Explore sales and customer value



In [6]:
gross_sales <- sum(retail_analysis$LineRevenue, na.rm = TRUE)
cat("Total recorded revenue:", round(gross_sales, 2), "\n")

product_leaders <- retail_analysis |>
  dplyr::group_by(StockCode, Description) |>
  dplyr::summarise(Revenue = sum(LineRevenue, na.rm = TRUE), .groups = "drop") |>
  dplyr::arrange(dplyr::desc(Revenue)) |>
  dplyr::slice_head(n = 5)

country_leaders <- retail_analysis |>
  dplyr::filter(!is.na(Country)) |>
  dplyr::group_by(Country) |>
  dplyr::summarise(Revenue = sum(LineRevenue, na.rm = TRUE), .groups = "drop") |>
  dplyr::arrange(dplyr::desc(Revenue)) |>
  dplyr::slice_head(n = 5)

customer_spend <- retail_analysis |>
  dplyr::filter(!is.na(CustomerID)) |>
  dplyr::group_by(CustomerID) |>
  dplyr::summarise(CustomerRevenue = sum(LineRevenue, na.rm = TRUE), .groups = "drop") |>
  dplyr::arrange(dplyr::desc(CustomerRevenue))

customer_leaders <- dplyr::slice_head(customer_spend, n = 5)

customer_spend <- customer_spend |>
  dplyr::mutate(Segment = dplyr::case_when(
    CustomerRevenue < 100 ~ "Low Value",
    CustomerRevenue < 500 ~ "Medium Value",
    CustomerRevenue < 1000 ~ "High Value",
    TRUE ~ "Premium"
  ))

cat("\nTop products by revenue\n"); print(product_leaders)
cat("\nTop countries by revenue\n"); print(country_leaders)
cat("\nTop customers by revenue\n"); print(customer_leaders)
cat("\nCustomer segment counts\n"); print(table(customer_spend$Segment))


Total recorded revenue: 10752840 

Top products by revenue
# A tibble: 5 × 3
  StockCode Description                        Revenue
  <chr>     <chr>                                <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE        168470.
2 47566     PARTY BUNTING                      142438.
3 22423     REGENCY CAKESTAND 3 TIER           135605.
4 85123A    WHITE HANGING HEART T-LIGHT HOLDER  93746.
5 23166     MEDIUM CERAMIC TOP STORAGE JAR      81033.

Top countries by revenue
# A tibble: 5 × 2
  Country         Revenue
  <chr>             <dbl>
1 United Kingdom 8861857.
2 Netherlands     363884.
3 EIRE            331660.
4 Germany         263819.
5 France          226976.

Top customers by revenue
# A tibble: 5 × 2
  CustomerID CustomerRevenue
       <dbl>           <dbl>
1      18102         408760.
2      14646         357531.
3      17450         186038.
4      14911         182691.
5      16446         168472.

Customer segment counts

  High Value    Low Value Medium Value 

## Save the integrated table in SQLite


In [7]:
database_connection <- DBI::dbConnect(RSQLite::SQLite(), "retail_analysis.sqlite")

DBI::dbWriteTable(database_connection, "integrated_sales",
                  retail_analysis, overwrite = TRUE)

sql_top_customers <- DBI::dbGetQuery(database_connection, "
  SELECT CustomerID, SUM(LineRevenue) AS RevenueTotal
  FROM integrated_sales
  WHERE CustomerID IS NOT NULL
  GROUP BY CustomerID
  ORDER BY RevenueTotal DESC
  LIMIT 5
")

sql_country_totals <- DBI::dbGetQuery(database_connection, "
  SELECT Country, SUM(LineRevenue) AS RevenueTotal
  FROM integrated_sales
  WHERE Country IS NOT NULL
  GROUP BY Country
  ORDER BY RevenueTotal DESC
")

cat("SQL: five highest-spending customers\n")
print(sql_top_customers)
cat("\nSQL: revenue by country\n")
print(sql_country_totals)

DBI::dbDisconnect(database_connection)
cat("\nDatabase saved as retail_analysis.sqlite\n")


SQL: five highest-spending customers
  CustomerID RevenueTotal
1      18102     408760.0
2      14646     357531.1
3      17450     186038.0
4      14911     182690.5
5      16446     168472.5

SQL: revenue by country
                Country RevenueTotal
1        United Kingdom   8861857.13
2           Netherlands    363884.48
3                  EIRE    331660.17
4               Germany    263818.97
5                France    226975.60
6             Australia    173918.61
7                 Spain     67426.09
8           Switzerland     66619.97
9                 Japan     48600.22
10              Belgium     47858.02
11               Sweden     43652.09
12               Norway     40283.20
13             Portugal     32679.45
14              Finland     23306.79
15      Channel Islands     22059.72
16              Denmark     21291.79
17                Italy     21065.45
18               Cyprus     16829.62
19            Singapore     11497.98
20               Poland     10264.54
21   

In [8]:
if (nrow(country_leaders) > 0) {
  strongest_country <- country_leaders$Country[1]
  cat("Highest-revenue country:", strongest_country, "\n")
}
if (nrow(product_leaders) > 0) {
  strongest_product <- product_leaders$Description[1]
  cat("Highest-revenue product:", strongest_product, "\n")
}

readr::write_csv(retail_analysis, "integrated_retail_output.csv")

cat("\nExport complete. Output files include:",
    "\n- sales_transactions.csv",
    "\n- catalog_items.json",
    "\n- customer_directory.xlsx",
    "\n- integrated_retail_output.csv",
    "\n- retail_analysis.sqlite\n")


Highest-revenue country: United Kingdom 
Highest-revenue product: PAPER CRAFT , LITTLE BIRDIE 

Export complete. Output files include: 
- sales_transactions.csv 
- catalog_items.json 
- customer_directory.xlsx 
- integrated_retail_output.csv 
- retail_analysis.sqlite


## Conclusion

The project successfully integrated and analyzed retail data using R and SQLite. After cleaning, 392,708 transaction records and 3,855 product records were retained, producing an integrated dataset of 387,877 rows and 9 columns. The analysis identified total revenue of approximately 10.75 million, with the United Kingdom as the highest-revenue country and PAPER CRAFT, LITTLE BIRDIE as the top product. Customer segmentation and SQL queries provided useful insights into purchasing patterns and spending behavior. Overall, the project demonstrated how data cleaning, integration, and analysis can transform raw retail data into meaningful insights for better business decision-making.